# Linear Regression Exam Practice Problems 🎯

## Same Mr Beast Dataset, Different Twists

**These practice problems mirror the style of the real quiz but change:**
- Target variable (predict `views` or `comments` instead of `likes`)
- Train/test split ratios (80/20, 75/25)
- Random seeds
- Predictor combinations
- Interaction terms

**Answer format**: Each section has `🎯 Question` → `💡 Hint` → `✅ Answer + Code`.

Try to solve each question yourself first, THEN check the answer!

## 🔧 Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score

data = pd.read_csv('mr_beast.csv')
data['publish_day'] = pd.Categorical(data['publish_day'],
                                      categories=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                                      ordered=True)
print('Data shape:', data.shape)

---
## 🎯 Practice 1.  Train/Test Split with an 80/20 ratio

Split the data into 80% train and 20% test using simple random sampling (`data.sample()`), `random_state=617`.

**Question**: What is the **mean** of `views` in the test sample?

💡 Hint: `test.views.mean()`

---

In [ ]:
train = data.sample(frac=0.8, random_state=617)
test = data.drop(labels=train.index)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Mean of views in test:', test.views.mean())

# ✅ Answer: 34287938.24444444

---
## 🎯 Practice 2. Correlation with `comments` (not likes!)

Using the **train** sample from Practice 1, find:

(a) Which variable has the **weakest** (smallest |r|) pearson correlation with `comments`?

(b) What is the p-value of the correlation between `duration_seconds` and `comments`?

(c) Is that correlation significant at α=0.05?

💡 Hint: Loop through `pearsonr()` for each variable.

---

In [ ]:
numeric_vars = ['views','likes','duration_seconds','number_of_tags',
                'length_description','length_title','time_since']

print(f'{"Variable":<22}{"r":>10}{"|r|":>10}{"p-value":>15}')
print('-'*57)
for var in numeric_vars:
    r, p = pearsonr(train[var], train['comments'])
    print(f'{var:<22}{r:>10.4f}{abs(r):>10.4f}{p:>15.4e}')

# ✅ (a) Weakest: length_description (or duration_seconds) depending on sample
# ✅ (b) & (c): check p-value; if < 0.05 → significant

---
## 🎯 Practice 3. Simple regression: `views ~ duration_seconds`

Use the 80/20 train sample from Practice 1.

(a) What is the coefficient of `duration_seconds`?

(b) Is it statistically significant?

(c) What is the R² of this model?

(d) Predict views for a video with `duration_seconds = 600`.

💡 Hint: `model = ols('views ~ duration_seconds', data=train).fit()`

---

In [ ]:
model_p3 = ols('views ~ duration_seconds', data=train).fit()
print(model_p3.summary())

print('\n(a) Coefficient:', model_p3.params['duration_seconds'])
print('(b) p-value:', model_p3.pvalues['duration_seconds'])
print('(c) R²:', model_p3.rsquared)
print('(d) Prediction for duration_seconds=600:',
      model_p3.predict(pd.DataFrame({'duration_seconds':[600]})).values[0])

---
## 🎯 Practice 4. Simple regression with a different reference level

What if we want **Sunday** to be the reference level for `publish_day` instead of Monday?

(a) Recategorize `publish_day` so that `Sun` is first.

(b) Run `likes ~ publish_day`.

(c) Which days are now significantly different from Sunday?

💡 Hint: Change the `categories=[...]` list order.

---

In [ ]:
# Make a copy so we don't break the original
train_p4 = train.copy()
train_p4['publish_day'] = pd.Categorical(
    train_p4['publish_day'].astype(str),
    categories=['Sun','Mon','Tue','Wed','Thu','Fri','Sat'],  # Sun first
    ordered=True)

model_p4 = ols('likes ~ publish_day', data=train_p4).fit()
print(model_p4.summary())

# ✅ Now the intercept = mean likes for Sunday
# ✅ Each coefficient = difference from Sunday

---
## 🎯 Practice 5. Multiple regression with an interaction

Back to 90/10 split, `random_state=617`, Monday as reference for `publish_day`.

Fit `model_int = ols('likes ~ time_since * number_of_tags', data=train).fit()`.

(a) Is the **interaction term** significant?

(b) What is the R² of this model?

(c) How would you interpret a significant interaction?

💡 Hint: `time_since * number_of_tags` expands to `time_since + number_of_tags + time_since:number_of_tags`.

---

In [ ]:
# Redo the 90/10 split
train90 = data.sample(frac=0.9, random_state=617)
test90 = data.drop(labels=train90.index)

model_int = ols('likes ~ time_since * number_of_tags', data=train90).fit()
print(model_int.summary())

print('\nInteraction p-value:',
      model_int.pvalues['time_since:number_of_tags'])
print('R²:', model_int.rsquared)

# ✅ (c) A significant interaction means the effect of time_since on likes
#    depends on number_of_tags (and vice versa).

---
## 🎯 Practice 6. Full multiple regression with all numeric predictors

Fit a model predicting `likes` using ALL numeric predictors:  
`views + comments + duration_seconds + number_of_tags + length_description + length_title + time_since`

(a) What is the R² of this model?

(b) Which predictors are NOT significant (p ≥ 0.05)?

(c) How do you interpret the `views` coefficient?

💡 Hint: Use string concatenation to build the formula.

---

In [ ]:
formula = ('likes ~ views + comments + duration_seconds + number_of_tags + '
           'length_description + length_title + time_since')
model_full = ols(formula, data=train90).fit()
print(model_full.summary())

print('\n(a) R²:', model_full.rsquared)
print('\n(b) Non-significant predictors (p ≥ 0.05):')
for name, p in model_full.pvalues.items():
    if p >= 0.05 and name != 'Intercept':
        print(f'   {name}: p = {p:.4f}')

# ✅ (c) Interpretation template:
# 'Holding all other variables constant, a 1-unit increase in views is
#  associated with a [coef] change in likes.'

---
## 🎯 Practice 7. Sklearn — predict `comments` with stratified split

Set `y = data.comments` and `X` = all other numeric + publish_day (drop video_id, title, description, publish_date, views, likes).

(a) Create `y_binned = pd.cut(y, bins=[-1,1000,10000,100000,1000000,10000000])`.

(b) Do a **75/25** stratified split with `random_state=617`.

(c) Dummy-encode `publish_day` with `drop_first=True`, then **align** test columns to train.

(d) Fit a linear regression and report the **test RMSE**.

---

In [ ]:
y = data.comments
X = data[['duration_seconds','number_of_tags','length_description',
          'length_title','time_since','publish_day']]

y_binned = pd.cut(y, bins=[-1,1000,10000,100000,1000000,10000000])
print('Bin counts:\n', y_binned.value_counts())

X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
    X, y, train_size=0.75, random_state=617, stratify=y_binned)

# Dummy encode
X_tr = pd.get_dummies(X_tr_raw, columns=['publish_day'], drop_first=True).astype(float)
X_te = pd.get_dummies(X_te_raw, columns=['publish_day'], drop_first=True).astype(float)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

reg = LinearRegression().fit(X_tr, y_tr)
rmse_test = root_mean_squared_error(y_te, reg.predict(X_te))
print('\nTest RMSE for comments:', rmse_test)

---
## 🎯 Practice 8. Coefficient interpretation — which sentence is correct?

Given `model = ols('likes ~ views + time_since', data=train).fit()`,  
and suppose `views` has coefficient `4.25`, `time_since` has coefficient `-10.5`.

Evaluate each statement:

**A.** Every additional view increases likes by 4.25.  
**B.** Holding `time_since` constant, every additional view is associated with a 4.25 increase in likes.  
**C.** If a video has 0 views and 0 time_since, likes will be 4.25.  
**D.** An additional hour since release decreases likes by 10.5, regardless of views.  
**E.** Holding `views` constant, every additional hour since release is associated with a 10.5 decrease in likes.

💡 **Correct: B and E.**  
✅ Multiple regression → always say *"holding other variables constant"*.

**A** and **D** skip this important detail → technically incorrect.  
**C** confuses intercept with a coefficient.

---

---
## 🎯 Practice 9. Predicted value & residual for a specific observation

Using `model1 = ols('likes ~ time_since', data=train).fit()` from the quiz (90/10 split),

(a) What is the predicted likes for the **10th** observation in train?

(b) What is the **residual** (actual − predicted) for that observation?

💡 Hint:  
- Predicted: `model1.predict(train.iloc[[9]])` (note `iloc[9]` is the 10th row — 0-indexed)  
- Actual: `train.iloc[9]['likes']`

---

In [ ]:
model1 = ols('likes ~ time_since', data=train90).fit()

# 10th observation (0-indexed, so iloc[9])
tenth = train90.iloc[[9]]
pred = model1.predict(tenth).values[0]
actual = tenth['likes'].values[0]

print('Predicted:', pred)
print('Actual:   ', actual)
print('Residual: ', actual - pred)

---
## 🎯 Practice 10. Compare train vs test RMSE for 4 models

Build 4 models predicting `likes` and compare their train and test RMSE:

- **Model A**: `likes ~ time_since`
- **Model B**: `likes ~ number_of_tags`
- **Model C**: `likes ~ time_since + number_of_tags`
- **Model D**: `likes ~ time_since * number_of_tags`

(a) Which model has the best TEST RMSE?

(b) Does the best train RMSE match the best test RMSE?

---

In [ ]:
def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))

formulas = {
    'A': 'likes ~ time_since',
    'B': 'likes ~ number_of_tags',
    'C': 'likes ~ time_since + number_of_tags',
    'D': 'likes ~ time_since * number_of_tags',
}

results = []
for name, f in formulas.items():
    m = ols(f, data=train90).fit()
    results.append({
        'model': name,
        'formula': f,
        'train_rmse': rmse(train90.likes, m.predict()),
        'test_rmse':  rmse(test90.likes, m.predict(test90)),
        'r2': m.rsquared,
    })

df_results = pd.DataFrame(results).set_index('model')
print(df_results)
print('\nBest train RMSE:', df_results.train_rmse.idxmin())
print('Best test RMSE: ', df_results.test_rmse.idxmin())

---
# 🎓 Exam Tips Summary

## Code patterns you MUST know by heart

```python
# 1. Random split
train = data.sample(frac=0.9, random_state=617)
test  = data.drop(labels=train.index)

# 2. Stratified split (sklearn)
y_binned = pd.cut(y, bins=[...])
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, train_size=0.9, random_state=617, stratify=y_binned)

# 3. Correlation with p-value
from scipy.stats import pearsonr
r, p = pearsonr(x, y)

# 4. OLS in statsmodels (auto-handles categoricals)
from statsmodels.formula.api import ols
m = ols('y ~ x1 + x2 + x1:x2', data=train).fit()
m.summary(); m.params; m.pvalues; m.rsquared; m.predict(new_df)

# 5. Sklearn pipeline
X_tr = pd.get_dummies(X_tr_raw, columns=['cat'], drop_first=True).astype(float)
X_te = pd.get_dummies(X_te_raw, columns=['cat'], drop_first=True).astype(float)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)  # ← CRUCIAL
reg = LinearRegression().fit(X_tr, y_tr)

# 6. RMSE
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_true, y_pred)
# Or manually:
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
```

## Common traps 🚨

1. **Dummy variable after get_dummies is boolean** → cast with `.astype(float)` or `.astype(int)`.
2. **Test set column alignment** — always `reindex(columns=X_train.columns, fill_value=0)` after get_dummies.
3. **Reference level** — `drop_first=True` drops the first category. For ordered Categorical, that's the first listed.
4. **Single predictor in sklearn** — use `[['col']]` (2D), not `['col']` (1D).
5. **"holding other variables constant"** — required when interpreting multi-predictor coefficients.
6. **"strongest correlation"** — use `abs(r)`, not r!  
7. **p-value vs coefficient** — p-value tells you SIGNIFICANCE; coefficient tells you EFFECT SIZE.
8. **Don't round answers** unless the question explicitly says to.

## Quick interpretation templates

- **Numeric, simple regression**: "A 1-unit increase in X is associated with a β-unit change in Y."
- **Numeric, multiple regression**: "Holding all other variables constant, a 1-unit increase in X is associated with a β-unit change in Y."
- **Dummy variable**: "Being in category K is associated with a β-unit difference in Y compared to the reference category."
- **Interaction (X₁·X₂ significant)**: "The effect of X₁ on Y depends on the value of X₂."
- **R² = 0.25**: "The model explains 25% of the variation in Y."
- **F-test p < 0.05**: "The overall model is statistically significant (at least one coefficient ≠ 0)."